Download the feature file **"features-clip-laion.tar.gz"** from Zenodo: [Zenodo Link](https://zenodo.org/records/8188570).
Run this notebook (about 15 minutes) to obtain the data you need to run both the demo and the time experiments

In [1]:
import h5py
import os
import numpy as np
from tqdm import tqdm  
import pickle
import h5py
import numpy as np
import os
from utils.f_files import *

In [ ]:
# merge the directories into a single file 
base_folder = 'features-clip-laion'

def combine_hdf5_files(base_folder):
    # Loop over all numbers from 1 to 17235
    for i in range(1, 17236):
        # Build the path of the HDF5 file using the number
        folder_name = f"{i:05d}"
        file_name = f"{i:05d}-clip-laion.hdf5"
        file_path = os.path.join(base_folder, folder_name, file_name)

        # Check if the file exists
        if os.path.exists(file_path):
            # Open the HDF5 file
            with h5py.File(file_path, 'r') as f:
                data = f['data'][:]
                ids = f['ids'][:]

                # Check if it is the first file; if so, create the combined file
                if i == 1:
                    try:
                        # Open the combined HDF5 file only if it is not already open in read-only mode
                        combined_f = h5py.File('combined.h5', 'w')
                    except OSError:
                        # If the file is already open in read-only mode, close it before reopening it in append mode
                        combined_f.close()
                        combined_f = h5py.File('combined.h5', 'a')

                    combined_data_shape = (0,) + data.shape[1:]
                    combined_ids_shape = (0,)

                    # Create the datasets in the combined HDF5 file with chunking enabled
                    combined_data = combined_f.create_dataset('data', shape=combined_data_shape, maxshape=(None,) + data.shape[1:], dtype='<f4')
                    combined_ids = combined_f.create_dataset('ids', shape=combined_ids_shape, maxshape=(None,), dtype=h5py.special_dtype(vlen=str))

                    # Copy the data from the first file
                    combined_data.resize((len(data),) + data.shape[1:])
                    combined_ids.resize((len(ids),))
                    combined_data[:] = data
                    combined_ids[:] = ids.astype(str)
                else:
                    # Add data to the dataset in the combined file
                    combined_data.resize((combined_data.shape[0] + len(data),) + combined_data.shape[1:])
                    combined_ids.resize((combined_ids.shape[0] + len(ids),))
                    combined_data[-len(data):, :] = data
                    combined_ids[-len(ids):] = ids.astype(str)

        print(f"File {file_path} combined successfully.")

    # Close the combined HDF5 file at the end
    combined_f.close()

combine_hdf5_files(base_folder)


In [ ]:
# index the file 

# Paths to the original and new HDF5 files

original_hdf5_file_path = "/home/francescascotti/dev/interfaccia/combined.h5"
new_hdf5_file_path = "/home/francescascotti/dev/interfaccia/indexed_file.h5"
# Verifica se il file originale esiste
if not os.path.exists(original_hdf5_file_path):
    raise FileNotFoundError(f"Il file originale '{original_hdf5_file_path}' non esiste.")

# Batch size
batch_size = 1000

# Delete the existing file if it exists
if os.path.exists(new_hdf5_file_path):
    os.remove(new_hdf5_file_path)

# Open the original HDF5 file
with h5py.File(original_hdf5_file_path, 'r') as original_file:
    # Read the 'data' and 'ids' groups
    data = original_file['data']
    ids = original_file['ids']

    # Calculate total number of data entries
    total_entries = len(ids)

    # Create a new HDF5 file to store the indexed data
    with h5py.File(new_hdf5_file_path, 'w') as new_file:
        # Create a group for the indexed data
        indexed_group = new_file.create_group('indexed_data')
        
        # Create datasets for ids and data
        id_dataset = indexed_group.create_dataset('ids', (total_entries,), maxshape=(None,), dtype=ids.dtype)
        data_dataset = indexed_group.create_dataset('data', (total_entries,) + data.shape[1:], maxshape=(None,) + data.shape[1:], dtype=data.dtype)

        # Store data entries in batches with a progress bar
        for i in tqdm(range(0, total_entries, batch_size), desc="Processing Batches"):
            batch_ids = ids[i:i+batch_size]
            batch_data = data[i:i+batch_size]

            # Write the batch to the datasets
            id_dataset[i:i+batch_size] = batch_ids
            data_dataset[i:i+batch_size] = batch_data

print("Indexed HDF5 file created successfully.")


Processing Batches: 100%|██████████| 2631/2631 [00:44<00:00, 58.48it/s]

Indexed HDF5 file created successfully.


In [ ]:

# Path to the HDF5 file
file_hdf5_path ="/home/francescascotti/dev/interfaccia/indexed_file.h5"

try:
    # Read the HDF5 file and create the index
    index_2_id, indexed_data, indexed_ids = read_indexed_hdf5_and_create_index(file_hdf5_path)
    print("Successfully read and indexed HDF5 file.")
    
    # Create logistic features
    indexed_data_logistic = create_logistic_indexed_data(indexed_data)
    print("Logistic features successfully created.")
    
    # Save all files locally with pickle dump
    files_to_save = {
        'index_2_id': index_2_id,
        'indexed_data': indexed_data,
        'indexed_ids': indexed_ids,
        'indexed_data_logistic': indexed_data_logistic
    }
    
    for file_name, data in files_to_save.items():
        with open(file_name, 'wb') as handle:
            pickle.dump(data, handle, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"Successfully saved {file_name}")

except Exception as e:
    print(f"An error occurred: {e}")




Successfully read and indexed HDF5 file.
Logistic features successfully created.
Successfully saved index_2_id2
Successfully saved indexed_data2
Successfully saved indexed_ids2
Successfully saved indexed_data_logistic
